In [ ]:
%pip install -U kagglehub ultralytics

In [ ]:
%pip uninstall -y opencv-python
%pip install -U opencv-python-headless

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("viacheslavshalamov/russian-road-signs-segmentation-dataset")

In [ ]:
import json
import os
import shutil
import yaml


def prepare_dataset(dataset_path: str, destination_dir: str) -> str:
    data = os.path.join(destination_dir, 'data.yaml')
    if os.path.exists(data):
        return data

    names = []
    for dir in ['train', 'val']:
        src_path = os.path.join(dataset_path, dir)
        images_path = os.path.join(destination_dir, dir, 'images')
        labels_path = os.path.join(destination_dir, dir, 'labels')
        os.makedirs(images_path, exist_ok=True)
        os.makedirs(labels_path, exist_ok=True)

        with open(os.path.join(src_path, 'via_region_data.json')) as f:
            region_data = json.load(f)

        for _, image in region_data.items():
            filename = image['filename']
            file_attributes = image['file_attributes']
            width = file_attributes['width']
            height = file_attributes['height']

            lines = []
            for _, region in image['regions'].items():
                region_attributes = region['region_attributes']
                name = region_attributes.get('name')
                if name is None:
                    print(f"Skipping region in '{filename}' (no name)")
                    continue
                if not name in names:
                    names.append(name)
                cls = names.index(name)

                shape_attributes = region['shape_attributes']
                xs = shape_attributes.get('all_points_x')
                ys = shape_attributes.get('all_points_y')
                if xs is None or ys is None:
                    print(f"Skipping region in '{filename}' (no points)")
                    continue

                points = " ".join(f"{x / width} {y / height}" for x, y in zip(xs, ys))
                lines.append(f"{cls} {points}\n")

            if len(lines) == 0:
                print(f"Skipping empty image '{filename}'")
                continue

            shutil.copy(os.path.join(src_path, filename), images_path)
            with open(os.path.join(labels_path, filename.replace(".jpg", ".txt")), 'w') as f:
                f.writelines(lines)


    data_content = {
        'train': 'train/images',
        'val': 'val/images',
        'names': names
    }
    with open(data, 'w') as f:
        yaml.dump(data_content, f, sort_keys=False)
    return data

In [ ]:
data = prepare_dataset(os.path.join(dataset_path, 'sign_dataset'), 'dataset')

In [ ]:
import torch

if torch.cuda.is_available():
    print("Using cuda")
    device = "cuda"
elif torch.mps.is_available():
    print("Using mps")
    device = "mps"
else:
    print("Using cpu")
    device = "cpu"

In [ ]:
def default_model(pretrained: bool=True) -> str:
    return 'yolo26n-seg.pt' if pretrained else 'yolo26n-seg.yaml'

def run_path(name: str) -> str:
    return os.path.join('runs', 'segment', name)

def model_path(name: str, file: str='best.pt') -> str:
    return os.path.join(run_path(name), 'weights', file)

def results_path(name: str) -> str:
    return os.path.join('results', name)

In [ ]:
from ultralytics import YOLO


def train(
    name: str,
    model: str,
    data: str,
    epochs: int,
):
    aug_params = {
        'mosaic': 0.0,
    }

    YOLO(model).train(
        data=data,
        epochs=epochs,
        cache="disk",
        device=device,
        workers=0,
        name=name,
        exist_ok=True,
        **aug_params,
    )

def resume_training(name: str):
    YOLO(model_path(name, 'last.pt')).train(resume=True)

In [ ]:
import cv2

from ultralytics import solutions


def do_instance_segmentation(
    instance_segmentation: solutions.InstanceSegmentation,
    capture: cv2.VideoCapture,
    writer: cv2.VideoWriter,
):
    while capture.isOpened():
        success, image = capture.read()
        if not success:
            break
        results = instance_segmentation(image)
        writer.write(results.plot_im)


def instance_segmantation(model: str, input: str, output: str, tracker: str):
    instance_segmentation = solutions.InstanceSegmentation(
        model=model,
        tracker=f'{tracker}.yaml',
        verbose=False,
        device=device,
    )

    capture = cv2.VideoCapture(input)
    try:
        w, h, fps = (int(capture.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
        writer = cv2.VideoWriter(output, cv2.VideoWriter.fourcc(*"mp4v"), fps, (w, h))
        try:
            do_instance_segmentation(instance_segmentation, capture, writer)
        finally:
            writer.release()
    finally:
        capture.release()

In [ ]:
import shutil


def save_results(name: str):
    os.makedirs(results_path(name))
    shutil.copy(
        os.path.join(run_path(name), 'results.png'),
        os.path.join(results_path(name), 'results.png'),
    )

def run_test_videos(name: str, videos_dir: str, tracker: str):
    model = model_path(name)
    dest = results_path(name)
    for video in os.listdir(videos_dir):
        print(f"Running '{tracker}' on '{video}'")
        instance_segmantation(
            model,
            os.path.join(videos_dir, video),
            os.path.join(dest, tracker, video),
            tracker,
        )

In [ ]:
def run_training(name: str, save: bool=True, **kwargs):
    if os.path.exists(model_path(name)):
        try:
            resume_training(name)
        except AssertionError as e:
            if 'nothing to resume' in str(e):
                print('Already finished training, skipping')
            else:
                raise e
    else:
        train(name, **kwargs)

    if not save:
        return

    if os.path.exists(results_path(name)):
        print('Already saved results, skipping')
    else:
        save_results(name)
        for tracker in ['botsort', 'bytetrack']:
            run_test_videos(name, 'videos', tracker)

In [ ]:
run_training(
    'train',
    model=default_model(),
    data=data,
    epochs=20,
)